# Layer Normalization

$$
y = \frac{x - E[x]}{\sqrt{Var[x] + \epsilon}} * \gamma + \beta
$$

Layer Norm chuẩn hoá theo **từng sample**, trên (các) chiều feature cuối cùng — khác với Batch Norm (chuẩn hoá theo chiều batch). `gamma` và `beta` là 2 tham số học được, dùng để scale/shift lại sau khi chuẩn hoá.

Notebook này viết hoàn toàn bằng `numpy`, không dùng PyTorch.

In [1]:
import numpy as np

### Ví dụ đơn giản

Giả sử có 1 batch gồm 2 câu (sentence), mỗi câu chỉ có 1 từ, mỗi từ được biểu diễn bởi vector 3 chiều (`embedding_dim = 3`).

In [2]:
inputs = np.array([[[0.2, 0.1, 0.3], [0.5, 0.1, 0.1]]])
inputs.shape  # (batch_size, sentence_length, embedding_dim)

(1, 2, 3)

In [3]:
# đưa sentence_length lên trước batch_size, giống convention (S, B, E)
B, S, E = inputs.shape
inputs = inputs.reshape(S, B, E)
inputs.shape

(2, 1, 3)

In [4]:
inputs

array([[[0.2, 0.1, 0.3]],

       [[0.5, 0.1, 0.1]]])

`gamma`, `beta` có shape bằng shape của (các) chiều cần chuẩn hoá — ở đây là `embedding_dim`.

In [5]:
parameter_shape = inputs.shape[-1:]
gamma = np.ones(parameter_shape)
beta = np.zeros(parameter_shape)
parameter_shape, gamma, beta

((3,), array([1., 1., 1.]), array([0., 0., 0.]))

In [6]:
# các chiều cần tính mean/var, tính từ phải sang trái theo số chiều của parameter_shape
dims = [-(i + 1) for i in range(len(parameter_shape))]
dims

[-1]

In [7]:
mean = inputs.mean(axis=tuple(dims), keepdims=True)
mean.shape, mean

((2, 1, 1), array([[[0.2       ]],

       [[0.23333333]]]))

In [8]:
epsilon = 1e-5
var = ((inputs - mean) ** 2).mean(axis=tuple(dims), keepdims=True)
std = np.sqrt(var + epsilon)
std.shape, std

((2, 1, 1), array([[[0.08171087]],

       [[0.18858832]]]))

In [9]:
y = (inputs - mean) / std
y

array([[[-3.39680324e-16, -1.22382734e+00,  1.22382734e+00]],

       [[ 1.41401473e+00, -7.07007365e-01, -7.07007365e-01]]])

In [10]:
out = gamma * y + beta
out

array([[[-3.39680324e-16, -1.22382734e+00,  1.22382734e+00]],

       [[ 1.41401473e+00, -7.07007365e-01, -7.07007365e-01]]])

### Đóng gói thành class

Giống các module khác trong repo (`Linear`, `MultiHeadAttention`, `PositionalEncoding`, ...), viết hoàn toàn bằng `numpy`.

In [11]:
class LayerNormalization:
    def __init__(self, parameters_shape, eps=1e-5):
        self.parameters_shape = parameters_shape
        self.eps = eps
        self.gamma = np.ones(parameters_shape)
        self.beta = np.zeros(parameters_shape)

    def forward(self, inputs):
        dims = tuple(-(i + 1) for i in range(len(self.parameters_shape)))
        mean = inputs.mean(axis=dims, keepdims=True)
        var = ((inputs - mean) ** 2).mean(axis=dims, keepdims=True)
        std = np.sqrt(var + self.eps)
        y = (inputs - mean) / std
        out = self.gamma * y + self.beta
        return out

    def __call__(self, inputs):
        return self.forward(inputs)

### Kiểm tra với input lớn hơn

In [12]:
np.random.seed(42)
batch_size = 3
sentence_length = 5
embedding_dim = 8
inputs = np.random.randn(sentence_length, batch_size, embedding_dim) * 10

layer_norm = LayerNormalization(inputs.shape[-1:])
out = layer_norm(inputs)
out.shape

(5, 3, 8)

In [13]:
# sau khi chuẩn hoá: mean ~ 0, std ~ 1 theo chiều embedding_dim
out.mean(axis=-1), out.std(axis=-1)

(array([[-5.55111512e-17, -2.77555756e-17,  0.00000000e+00],
       [-2.77555756e-17,  2.77555756e-17, -5.55111512e-17],
       [-8.32667268e-17,  0.00000000e+00,  5.55111512e-17],
       [ 0.00000000e+00,  2.77555756e-17, -5.55111512e-17],
       [-1.11022302e-16,  0.00000000e+00,  2.77555756e-17]]),
 array([[0.99999989, 0.99999992, 0.99999994],
       [0.99999993, 0.99999994, 0.99999991],
       [0.99999994, 0.99999989, 0.99999989],
       [0.99999997, 0.99999991, 0.9999999 ],
       [0.99999982, 0.99999995, 0.99999995]]))